# Smart City Traffic Dakar — Analyse Spark + Iceberg

Ce notebook :
1. Lit les événements de trafic depuis **Kafka** (batch)
2. Les écrit dans une table **Iceberg** sur **MinIO**
3. Calcule des KPIs (vitesse, densité, alertes)
4. Démontre les principales **techniques d'optimisation Spark**

> Le script standalone `scripts/spark_kafka_to_iceberg.py` contient la même logique
> et peut être déclenché depuis Airflow (DAG `spark_kafka_to_iceberg`).

## 1. Imports

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, avg, sum as spark_sum,
    broadcast, count, window
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
from pyspark import StorageLevel

## 2. Session Spark — configuration avec optimisations activées

Les variables d'environnement sont injectées par Docker Compose (ou Airflow).
Les optimisations **AQE** sont activées dès la création de la session.

In [ ]:
KAFKA_SERVERS  = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:9092")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_USER     = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_PASSWORD = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
ICEBERG_BUCKET = os.getenv("MINIO_ICEBERG_BUCKET", "traffic-bucket")

spark = (
    SparkSession.builder
    .appName("SmartCityTrafficAnalysis")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    # Catalogue Iceberg — sous-chemin /iceberg dans le bucket unique
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.my_catalog.type", "hadoop")
    .config("spark.sql.catalog.my_catalog.warehouse", f"s3a://{ICEBERG_BUCKET}/iceberg")
    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_USER)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_PASSWORD)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # ── Optimisation 1 : AQE (Adaptive Query Execution) ─────────────────────
    # Spark réoptimise le plan d'exécution selon les stats réelles à l'exécution
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    # Note: spark.sql.shuffle.partitions doit rester un entier (défaut=200)
    # AQE le réduira automatiquement via coalescePartitions
    .getOrCreate()
)
print("Session Spark prête.", spark.version)

## 3. Schéma des événements

In [ ]:
SCHEMA = StructType([
    StructField("event_id",      IntegerType(), True),
    StructField("timestamp",     StringType(),  True),
    StructField("city",          StringType(),  True),
    StructField("section_id",    StringType(),  True),
    StructField("section_name",  StringType(),  True),
    StructField("direction",     StringType(),  True),
    StructField("latitude",      DoubleType(),  True),
    StructField("longitude",     DoubleType(),  True),
    StructField("speed",         DoubleType(),  True),
    StructField("vehicle_count", IntegerType(), True),
])

## 4. Lecture batch depuis Kafka

In [ ]:
raw_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_SERVERS)
    .option("subscribe", "traffic_events")
    .option("startingOffsets", "earliest")
    .option("endingOffsets",   "latest")
    .option("failOnDataLoss",  "false")
    .load()
)
print(f"{raw_df.count()} messages bruts lus depuis Kafka.")

## 5. Parsing + filtrage

**Optimisation 2 — Column pruning** : on ne sélectionne que les colonnes utiles.

**Optimisation 3 — Predicate pushdown** : on filtre le plus tôt possible
pour réduire le volume traité dans les étapes suivantes.

In [ ]:
df = (
    raw_df
    # Column pruning : on ne garde que "value" avant le parsing
    .select(from_json(col("value").cast("string"), SCHEMA).alias("d"))
    .select("d.*")
    # Predicate pushdown : filtres dès la sortie du parsing
    .filter(col("city").isNotNull() & col("speed").isNotNull())
)
df.printSchema()
df.show(5)

## 6. Cache / Persist

**Optimisation 4 — Cache** : `df` est utilisé dans plusieurs transformations.
On le persiste pour éviter de relire depuis Kafka à chaque action.

- `cache()` = `MEMORY_AND_DISK` par défaut
- Utiliser `StorageLevel.DISK_ONLY` si la RAM est limitée

In [ ]:
df.persist(StorageLevel.MEMORY_AND_DISK)
print(f"DataFrame persisté — {df.count()} événements valides.")

## 7. Écriture dans Iceberg (MinIO)

In [ ]:
(
    df.write
    .format("iceberg")
    .mode("append")
    .save("my_catalog.default.traffic_events")
)
print("Écriture Iceberg terminée.")

## 8. KPIs

In [ ]:
# Vitesse moyenne par ville
df.groupBy("city").agg(avg("speed").alias("vitesse_moy")).orderBy("vitesse_moy").show()

In [ ]:
# Densité de véhicules par section
df.groupBy("section_id").agg(spark_sum("vehicle_count").alias("total_vehicules")).orderBy("total_vehicules", ascending=False).show()

In [ ]:
# Alertes : sections où la vitesse dépasse 90 km/h
df.filter(col("speed") > 90).select("event_id", "city", "section_id", "speed").show()

## 9. Optimisations avancées Spark

Cette section illustre les principales techniques d'optimisation disponibles.

### 9.1 Vérification de la configuration AQE

AQE permet à Spark de :
- **fusionner les petites partitions** après un shuffle
- **gérer les skew joins** (partitions déséquilibrées)
- **choisir dynamiquement** le type de join (broadcast vs sort-merge)

In [ ]:
conf = spark.conf
print("AQE activé                :", conf.get("spark.sql.adaptive.enabled"))
print("Coalesce partitions       :", conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
print("Gestion skew joins        :", conf.get("spark.sql.adaptive.skewJoin.enabled"))
print("Shuffle partitions (auto) :", conf.get("spark.sql.shuffle.partitions"))

### 9.2 Repartitionnement par colonne clé

**Optimisation 5 — Repartition** : regrouper les données par la colonne
utilisée pour les agrégations (`city`) évite des shuffles supplémentaires.

- `repartition(n, col)` : redistribue les données (shuffle réseau)
- `coalesce(n)` : réduit le nombre de partitions **sans shuffle** (plus rapide après filtrage)

In [ ]:
# Repartition par city : utile si on fait beaucoup d'agrégations groupées par ville
df_by_city = df.repartition(8, "city")
print("Partitions après repartition :", df_by_city.rdd.getNumPartitions())

# Coalesce après un filtre fort (ex: alertes) pour éviter des petites partitions vides
df_alerts = df.filter(col("speed") > 90).coalesce(2)
print("Partitions alertes coalesced :", df_alerts.rdd.getNumPartitions())

### 9.3 Broadcast Join

**Optimisation 6 — Broadcast join** : si un des deux DataFrames est petit
(table de référence, lookup), on le diffuse sur tous les exécuteurs pour
éviter un shuffle coûteux des deux côtés.

Seuil par défaut : `spark.sql.autoBroadcastJoinThreshold = 10 MB`.
AQE peut le déclencher automatiquement au-delà de ce seuil.

In [ ]:
# Table de référence petite : noms complets des sections
section_labels = spark.createDataFrame([
    ("SEC-100", "Autoroute à péage"),
    ("SEC-101", "Route de Rufisque"),
    ("SEC-102", "VDN Nord"),
    ("SEC-104", "Corniche Ouest"),
    ("SEC-110", "Route de Thiès"),
], ["section_id", "label"])

# broadcast() force la diffusion même si AQE ne la déclenche pas automatiquement
df_enrichi = df.join(broadcast(section_labels), on="section_id", how="left")
df_enrichi.select("city", "section_id", "label", "speed").show(5)

### 9.4 Plan d'exécution

**Optimisation 7 — Explain** : lire le plan physique permet de vérifier que
les optimisations sont bien appliquées (présence de `BroadcastHashJoin`,
`Filter` poussé tôt, `ColumnarToRow`, etc.).

In [ ]:
# Plan logique + physique
df_enrichi.explain(mode="formatted")

### 9.5 Compaction + Z-ordering Iceberg

**Optimisation 8 — Z-ordering** : réorganise les fichiers Iceberg pour que
les données filtrées par `city` ET `section_id` soient proches physiquement.
Réduit fortement le nombre de fichiers lus pour les requêtes analytiques.

**Compaction** (`rewrite_data_files`) : fusionne les petits fichiers
créés par des appends fréquents.

In [ ]:
# Compaction simple (fusionne les petits fichiers)
spark.sql("""
    CALL my_catalog.system.rewrite_data_files(
        table => 'default.traffic_events',
        strategy => 'sort',
        sort_order => 'city ASC, section_id ASC'
    )
""").show(truncate=False)

In [ ]:
# Suppression des anciens snapshots Iceberg (TTL = 7 jours)
spark.sql("""
    CALL my_catalog.system.expire_snapshots(
        table => 'default.traffic_events',
        older_than => TIMESTAMP '2026-01-01 00:00:00'
    )
""").show()

## 10. Nettoyage

In [ ]:
df.unpersist()
spark.stop()
print("Session Spark fermée.")